# OCR + RAG: Os Olhos Alimentando a Memória

In [28]:
# 1. Dependências do sistema para o Ollama detectar a GPU e descompactar arquivos
!apt-get update
!apt-get install -y zstd pciutils lshw

# 2. Instalação do Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Instalação das bibliotecas Python (Os PIPs)
!pip install easyocr pymupdf ollama langchain langchain-community langchain-ollama chromadb
!pip install --upgrade langchain langchain-core langchain-community langchain-text-splitters langchain-ollama chromadb

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [6]:
!pip install langchain-classic langchain-core

In [4]:
!nohup ollama serve > ollama.log 2>&1 &
!sleep 5

In [29]:

# Baixa os três modelos necessários (Isso vai baixar aprox. 10GB no total)
!ollama pull llava
!ollama pull nomic-embed-text
!ollama pull llama3

In [7]:
import io
import easyocr
import fitz
import ollama
import base64
import numpy as np
from pathlib import Path
from PIL import Image
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_classic.chains import RetrievalQA
from langchain_core.documents import Document

# Configurado para usar GPU no Colab
reader = easyocr.Reader(["pt", "en"], gpu=True)

def redimensionar_para_llava(caminho: str, max_size=(1024, 1024)) -> str:
    """Reduz o peso da imagem para evitar erro de memória no Ollama."""
    with Image.open(caminho) as img:
        if img.mode != 'RGB': img = img.convert('RGB')
        img.thumbnail(max_size, Image.Resampling.LANCZOS)
        buffer = io.BytesIO()
        img.save(buffer, format="JPEG", quality=85)
        return base64.b64encode(buffer.getvalue()).decode()

def digitalizar_documento(caminho: str) -> Document:
    caminho = Path(caminho)
    extensao = caminho.suffix.lower()

    if extensao == ".pdf":
        with fitz.open(caminho) as doc:
            tem_texto = any(p.get_text().strip() for p in doc)
        if tem_texto:
            texto = "\n".join(p.get_text() for p in fitz.open(caminho))
            metodo = "pdf_digital"
        else:
            paginas = []
            with fitz.open(caminho) as doc:
                for pagina in doc:
                    pix = pagina.get_pixmap(matrix=fitz.Matrix(2, 2))
                    img = Image.open(io.BytesIO(pix.tobytes("png")))
                    resultados = reader.readtext(np.array(img))
                    paginas.append(" ".join([t for _, t, c in resultados if c > 0.3]))
            texto = "\n\n".join(paginas)
            metodo = "pdf_ocr"

    elif extensao in [".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tiff"]:
        resultados = reader.readtext(str(caminho))
        texto_easyocr = " ".join([t for _, t, c in resultados if c > 0.4])

        if len(texto_easyocr) < 50:
            img_b64 = redimensionar_para_llava(str(caminho))
            resposta = ollama.chat(
                model="llava",
                messages=[{"role": "user", "content": "Transcreva o texto desta imagem.", "images": [img_b64]}]
            )
            texto = resposta["message"]["content"]
            metodo = "llava_vision"
        else:
            texto = texto_easyocr
            metodo = "easyocr"

    return Document(page_content=texto, metadata={"source": str(caminho), "metodo_ocr": metodo})

def construir_base_com_ocr(caminhos: list):
    documentos = []
    for caminho in caminhos:
        if Path(caminho).exists():
            doc = digitalizar_documento(caminho)
            documentos.append(doc)
            print(f"✅ {caminho} processado via {doc.metadata['metodo_ocr']}")

    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunks = splitter.split_documents(documentos)

    embeddings = OllamaEmbeddings(model="nomic-embed-text")
    db = Chroma.from_documents(chunks, embeddings)

    llm = ChatOllama(model="llama3", temperature=0.1)
    return RetrievalQA.from_chain_type(
        llm=llm,
        retriever=db.as_retriever(),
        return_source_documents=True
    )

# Execução
arquivos = ["/content/ocr-contas-enel (1).webp"] # Certifique-se que o arquivo está aqui
rag = construir_base_com_ocr(arquivos)

pergunta = "Qual o valor total da conta e a data de vencimento?"
res = rag.invoke(pergunta)
print(f"\n❓ Pergunta: {pergunta}")
print(f"🤖 Resposta: {res['result']}")

Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete✅ /content/ocr-contas-enel (1).webp processado via easyocr

❓ Pergunta: Qual o valor total da conta e a data de vencimento?
🤖 Resposta: According to the provided context, the value of the account is not explicitly stated. However, it mentions "Tarfas aplicadas (sem impostos) {TUSD) Leiturs" which suggests that there are tariffs applied without taxes.

The document also mentions "VENCI Nª DA 20 JUN 2022 NSC. EST: MAI2022", which indicates that the account is due on June 20, 2022.
